# Extractor de Costeo Unitario desde PDFs — DeepSeek + OCR local

**Documentación metodológica del script de extracción de costeo unitario (variante DeepSeek)**

Equipo de Costeo PDET — ART

---

## Objetivo del documento

Esta es la variante del extractor de costeo unitario que usa **DeepSeek** en
vez de Claude, con **OCR local (Tesseract)** como respaldo automático cuando
el PDF no tiene texto seleccionable (documentos escaneados). A diferencia de
Claude, DeepSeek no lee el PDF de forma nativa — solo recibe texto — así que
este script resuelve esa limitación haciendo el OCR **en la propia máquina**,
sin depender de ninguna capacidad del proveedor de LLM para eso.

> Igual que el extractor con Claude, esta versión corre sobre **el conjunto
> pleno de indicadores**, no sobre un piloto acotado: cualquier carpeta
> `Descargas/{codigo_indicador}/{codigo_contrato}.pdf` sirve de entrada,
> para tantos indicadores como se necesite procesar en una misma corrida.

## Flujo general

1. Por cada PDF, intenta extraer **texto nativo** con `pypdf`.
2. Si el texto es insuficiente (PDF escaneado), aplica **OCR local en dos
   pasadas**: un barrido rápido de baja resolución en todo el documento para
   ubicar las páginas con tablas de presupuesto/cantidades, y luego OCR de
   alta resolución **solo en esas páginas**.
3. El texto resultante (nativo u OCR) se manda a **DeepSeek** para extraer
   los campos de costeo unitario — mismo prompt y mismos 15 campos que la
   versión con Claude.

## Requisitos

```
py -m pip install openai pypdf pandas openpyxl pymupdf pytesseract pillow --break-system-packages
```

Además hay que instalar el motor **Tesseract OCR** (no es un paquete de
pip): <https://github.com/UB-Mannheim/tesseract/wiki>. Hay que ajustar
`RUTA_TESSERACT` en la celda de configuración con la ruta real de la
instalación.

> **Generalización de la base de entrada**
>
> Igual que en la versión con Claude: una carpeta con PDFs nombrados como el
> código del contrato, organizados en subcarpetas por indicador
> (`Descargas/{codigo_indicador}/`). El código del indicador se infiere del
> nombre de la subcarpeta — no depende de ninguna lista fija de
> indicadores dentro del script.

## Panorama general del pipeline

```
Carpeta de PDFs (Descargas/{codigo_indicador}/{codigo_contrato}[.pdf|_AT.pdf])
        │
        ▼
Carga de resultados previos (reanudación)
        │
        ▼
Por cada PDF pendiente:
  ¿tiene texto nativo suficiente? ──sí──► usar texto nativo
        │no
        ▼
  OCR pasada 1 (baja resolución, TODAS las páginas) → detectar páginas relevantes
        │
        ▼
  OCR pasada 2 (alta resolución, SOLO páginas relevantes + portada)
        │
        ▼
  Texto (nativo u OCR) ──► DeepSeek (mismo prompt de 15 campos)
        │
        ▼
  Parseo JSON con reintentos + cálculo de respaldo de costo_unitario_cop
        │
        ▼
Checkpoint cada N PDFs + Excel final con formato condicional por confianza
```

## Control global de warnings y errores

Misma celda reutilizable de los notebooks anteriores del pipeline: modificable por celda, ya sea cambiando las variables globales o sobreescribiendo `warnings.filterwarnings(...)` puntualmente.

In [ ]:
# ── Control global de warnings y errores (modificable por celda) ────────
import warnings

MOSTRAR_WARNINGS = False   # -> True para ver warnings de pandas/openpyxl/pytesseract aquí
DETENER_EN_ERROR = False   # -> True para propagar errores inesperados en vez de solo loguearlos

if MOSTRAR_WARNINGS:
    warnings.filterwarnings("default")
else:
    warnings.filterwarnings("ignore")

# Para reactivar warnings SOLO en una celda puntual (sin afectar el resto):
#   with warnings.catch_warnings():
#       warnings.filterwarnings("default")
#       ... código a depurar ...

## Configuración global

In [ ]:
import os
import re
import sys
import json
import time
import logging
import io as _io
from pathlib import Path
from datetime import datetime
from typing import Optional

import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

In [ ]:
PROVEEDOR = "deepseek"
MODELO    = "deepseek-chat"   # V3 — rápido y barato, bueno para extracción estructurada
# MODELO  = "deepseek-reasoner"  # R1 — más lento y ~2x más caro, úsalo si deepseek-chat
                                  # falla mucho en documentos ambiguos

# Ruta al ejecutable de Tesseract OCR (AJUSTA ESTO a tu instalación real)
RUTA_TESSERACT = r"C:\Program Files\Tesseract-OCR\tesseract.exe" 

In [ ]:
# ── ENTRADA (GENERAL) ─────────────────────────────────────────────────────
# Carpeta con PDFs organizados en subcarpetas por indicador:
#   Descargas/{codigo_indicador}/{codigo_contrato}.pdf
RUTA_PDFS_DEFAULT = "Descargas"
EXCEL_SALIDA      = "costeo_unitario_deepseek.xlsx"
LOG_FILE          = "extraccion_costeo_deepseek.log"

MAX_TOKENS_RESPUESTA   = 2000
MAX_TAMANO_MB          = 30
PAUSA_ENTRE_LLAMADAS   = 2      # DeepSeek es más barato/rápido, no necesitas pausas largas
MAX_REINTENTOS         = 5
PAUSA_RATE_LIMIT       = 60
CHECKPOINT_CADA        = 5
LIMITE_CHARS_TEXTO_PDF = 120_000   # tope de texto enviado al modelo
MAX_PAGINAS_OCR        = 12        # tope de páginas RELEVANTES que reciben OCR de alta
                                    # resolución (después del barrido rápido de detección)
DPI_OCR                = 200       # resolución del render para OCR de alta calidad

### Logging

In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[logging.FileHandler(LOG_FILE, encoding="utf-8"), logging.StreamHandler(sys.stdout)],
)
log = logging.getLogger(__name__)

## Fase 1 — Prompt de extracción

Mismos 15 campos que la versión con Claude (ver esa documentación para el detalle campo por campo). La única diferencia de contenido está en la instrucción de `confianza` y `observaciones`, que aquí también contemplan explícitamente errores de reconocimiento de caracteres cuando el texto viene de OCR.

In [ ]:
PROMPT_TEMPLATE = """Eres un analista experto en contratación pública colombiana (SECOP-II). \
Tu tarea es leer el documento contractual adjunto (Estudio Previo, Anexo Técnico o \
similar) y extraer la información clave para calcular el COSTEO UNITARIO del contrato.

CONTEXTO CRÍTICO — CÓDIGO DEL CONTRATO
──────────────────────────────────────
El nombre del archivo PDF que estás analizando corresponde al CÓDIGO DEL CONTRATO en SECOP-II:

    CÓDIGO_CONTRATO_ARCHIVO: {codigo_archivo}

Usa este código como referencia principal de identificación. Si dentro del documento aparecen \
identificadores adicionales (número interno tipo CCFV-065-2024, número de proceso de selección \
tipo LP-001-2024, etc.), inclúyelos en el campo `codigo_contrato` concatenándolos con " | " al \
código anterior.

CAMPOS A EXTRAER
────────────────
Extrae los siguientes campos. Si un dato NO aparece explícitamente en el documento, \
pon `null` (NO inventes valores).

1. **codigo_contrato**: identificadores del contrato encontrados EN el documento (número \
interno, número de proceso, etc.). Si aparecen varios, concaténalos con " | ". null si no \
encuentras ninguno en el texto.

2. **descripcion_contrato**: objeto del contrato tal como aparece en el documento \
(texto completo del objeto, máximo 500 caracteres).

3. **municipio**: municipio(s) o lugar(es) específicos de ejecución del contrato. \
Si son varios, sepáralos con "; ". Usa los nombres exactos como aparecen en el documento.

4. **departamento**: departamento(s) correspondiente(s) (Colombia). Separa con "; " si son varios.

5. **subregion_impacto**: si el documento menciona explícitamente una subregión PDET \
(ej: "Alto Patía - Norte del Cauca", "Catatumbo", "Sur de Bolívar", "Macarena - Guaviare") \
o una región de impacto específica, indícala. null si no aparece.

6. **year_contrato**: año del contrato (formato YYYY). Si aparece fecha completa, extrae \
el año. Si hay varias fechas (firma, inicio, terminación), usa la de FIRMA o SUSCRIPCIÓN. \
null si no aparece.

7. **precio_cop**: valor total del contrato en pesos colombianos (COP), como número entero \
sin puntos, comas ni símbolo $. Ejemplo: 7035424595. Si el valor está en otra moneda, \
extráelo tal cual y anota la moneda en `moneda`.

8. **moneda**: "COP", "USD", "EUR", etc. Por defecto "COP".

9. **cantidad**: ⚠️ CAMPO CRÍTICO — cantidad física que el contrato entrega/ejecuta, en su \
unidad natural. Busca con detenimiento en el objeto, alcance, obligaciones específicas, \
anexo técnico, presupuesto detallado, APUs y tablas de cantidades. \
Ejemplos: 683600 (hectáreas), 12.5 (km de vía), 240 (familias beneficiarias), \
177 (viviendas mejoradas), 4 (infraestructuras de salud: 1 centro + 3 puestos), \
150 (cupos de formación), 35 (organizaciones apoyadas), 8 (kits entregados). \
Usa número decimal con punto. null SOLO si después de revisar el documento completo \
realmente no aparece ninguna cantidad física asociada al producto/servicio.

10. **unidad_cantidad**: ⚠️ CAMPO CRÍTICO — unidad física de la cantidad (ej: "hectárea", \
"km", "familia", "vivienda", "vivienda mejorada", "centro de salud", "cupo de formación", \
"estudiante", "kit entregado", "metro lineal"). Debe ser coherente con `cantidad`. null si \
no aparece.

11. **costo_unitario_cop**: costo por unidad en COP. Si el documento lo reporta explícitamente \
(p. ej. en el APU o presupuesto), úsalo. Si no, calcúlalo como `precio_cop / cantidad`. \
Redondea a entero. null si no se puede calcular (faltan datos).

12. **componentes_costo**: desglose de los componentes del costo si el documento lo reporta \
— por ejemplo materiales, mano de obra, AIU (Administración, Imprevistos, Utilidad), \
transporte, interventoría, dotación. Texto breve tipo "Materiales 45%, Mano de obra 30%, \
AIU 15%, Transporte 10%" o una lista de los ítems principales del presupuesto/APU con su \
peso aproximado si el documento lo permite. null si el documento no desglosa el costo.

13. **fuente_cantidad**: nota breve de DÓNDE dentro del documento encontraste la cantidad \
— por ejemplo "Tabla de cantidades de obra, pág. 14", "Anexo técnico, ítem 3", "Cláusula \
segunda - objeto", "Presupuesto detallado, fila 'Vivienda mejorada'". Esto es para poder \
auditar la extracción después. null si no aplica.

14. **confianza**: tu confianza en la extracción — "alta", "media" o "baja". \
"baja" si el documento es ambiguo, está escaneado con mala resolución (o el texto viene de \
OCR con muchos errores evidentes), o no encontraste la mayoría de los campos clave \
(en especial `cantidad` y `precio_cop`).

15. **observaciones**: nota del analista (máximo 300 caracteres). Por ejemplo: \
"Contrato incluye interventoría dentro del valor"; "Cantidad aproximada"; \
"El documento es una adición, no el contrato original"; "No especifica cantidad, solo valor total"; \
"Cantidad inferida del anexo técnico tabla 4"; "Texto obtenido por OCR, verificar cifras".

FORMATO DE RESPUESTA
────────────────────
Responde ÚNICAMENTE con un objeto JSON válido. Sin texto antes ni después, sin backticks, \
sin explicaciones:

{{
  "codigo_contrato": "...",
  "descripcion_contrato": "...",
  "municipio": "...",
  "departamento": "...",
  "subregion_impacto": "...",
  "year_contrato": 2024,
  "precio_cop": 0,
  "moneda": "COP",
  "cantidad": 0.0,
  "unidad_cantidad": "...",
  "costo_unitario_cop": 0,
  "componentes_costo": "...",
  "fuente_cantidad": "...",
  "confianza": "alta",
  "observaciones": "..."
}}
"""

## Fase 2 — Lectura de PDF: texto nativo y OCR

In [ ]:
def extraer_texto_pdf(path_pdf: Path) -> str:
    """Extrae texto seleccionable del PDF con pypdf. Vacío si es un escaneo."""
    from pypdf import PdfReader
    try:
        reader = PdfReader(str(path_pdf))
    except Exception as e:
        log.warning(f"  No se pudo abrir el PDF con pypdf: {e}")
        return ""
    partes = []
    for page in reader.pages:
        try:
            partes.append(page.extract_text() or "")
        except Exception:
            continue
    return "\n".join(partes)

In [ ]:
def ocr_pdf(path_pdf: Path, dpi: int = None, max_paginas: int = None) -> str:
    """OCR local con PyMuPDF + Tesseract, en dos pasadas:
    1) barrido rápido y de baja resolución en TODO el documento para ubicar
       páginas relevantes (presupuesto, cantidades, valor del contrato).
    2) OCR de alta resolución solo en esas páginas + la primera (objeto/portada).
    No consume tokens de ningún proveedor — corre 100% en tu máquina.
    'max_paginas' limita cuántas páginas relevantes, como máximo, entran a
    la pasada HD (ver MAX_PAGINAS_HD abajo)."""
    import fitz
    import pytesseract
    from PIL import Image

    dpi = dpi or DPI_OCR

    PALABRAS_CLAVE = [
        "presupuesto", "valor total", "valor del contrato", "cantidad",
        "unidad de medida", "apu", "item", "ítem", "precio unitario",
        "análisis de precios", "cuadro de cantidades", "$", "cop",
    ]
    MAX_PAGINAS_HD = max_paginas or MAX_PAGINAS_OCR

    try:
        doc = fitz.open(str(path_pdf))
    except Exception as e:
        log.warning(f"  No se pudo abrir el PDF para OCR: {e}")
        return ""

    n_total = len(doc)
    log.info(f"  OCR — barrido rápido de {n_total} páginas para ubicar tablas relevantes...")

    # ── Pasada 1: rápida, baja resolución (dpi bajo = mucho más rápida) ──
    mat_rapida = fitz.Matrix(100 / 72, 100 / 72)
    puntajes = []
    for i in range(n_total):
        try:
            pix = doc[i].get_pixmap(matrix=mat_rapida)
            img = Image.open(_io.BytesIO(pix.tobytes("png")))
            texto = pytesseract.image_to_string(img, lang="spa", config="--psm 6")
            texto_low = texto.lower()
            score = sum(texto_low.count(k) for k in PALABRAS_CLAVE)
            puntajes.append((i, score))
        except Exception:
            puntajes.append((i, 0))

    # páginas relevantes: con score > 0, más la primera página siempre
    relevantes = {0}
    for i, score in puntajes:
        if score > 0:
            relevantes.add(i)

    # tope de seguridad para no hacer OCR de alta resolución en documentos
    # donde "todo" dio positivo por error (p.ej. contratos muy repetitivos)
    if len(relevantes) > MAX_PAGINAS_HD:
        top = sorted(puntajes, key=lambda x: -x[1])[:MAX_PAGINAS_HD]
        relevantes = {0} | {i for i, _ in top}

    log.info(f"  OCR — {len(relevantes)}/{n_total} páginas identificadas como relevantes: {sorted(relevantes)}")

    # ── Pasada 2: alta resolución, solo páginas relevantes ──
    zoom = dpi / 72
    mat_hd = fitz.Matrix(zoom, zoom)
    partes = []
    for i in sorted(relevantes):
        try:
            pix = doc[i].get_pixmap(matrix=mat_hd)
            img = Image.open(_io.BytesIO(pix.tobytes("png")))
            texto = pytesseract.image_to_string(img, lang="spa")
            partes.append(f"[Página {i + 1}]\n{texto}")
        except Exception as e:
            log.warning(f"  OCR HD falló en página {i + 1}: {e}")
            continue

    doc.close()
    return "\n\n".join(partes)

In [ ]:
def extraer_texto_o_ocr(path_pdf: Path) -> tuple:
    """Devuelve (texto, metodo). metodo es 'nativo' u 'ocr'.
    Intenta primero lectura nativa (gratis e instantánea); si el PDF resulta
    ser un escaneo (poco o ningún texto seleccionable), cae a OCR local."""
    texto = extraer_texto_pdf(path_pdf)
    texto_limpio = texto.strip()
    digitos = sum(c.isdigit() for c in texto_limpio)

    if len(texto_limpio) >= 500 and digitos >= 20:
        return texto, "nativo"

    log.info("  PDF sin texto seleccionable suficiente — aplicando OCR local (puede tardar)...")
    texto_ocr = ocr_pdf(path_pdf)
    if texto_ocr.strip():
        return texto_ocr, "ocr"
    return "", "ninguno"

Puntos clave de la estrategia de OCR en dos pasadas:

- **Pasada 1 (barrido rápido, ~100 DPI)**: corre sobre **todas** las páginas del documento, buscando palabras clave de presupuesto/cantidades (`presupuesto`, `valor total`, `apu`, `item`, `cop`, `$`, ...). Es deliberadamente de baja resolución — el objetivo no es leer bien, solo **ubicar** qué páginas probablemente tienen la tabla que importa.
- **Pasada 2 (alta resolución, 200 DPI por defecto)**: se aplica **solo** a las páginas que puntuaron en la pasada 1 (más la primera página, que casi siempre tiene el objeto del contrato). Esto evita hacer OCR de alta calidad —lento— en un documento de 80 páginas cuando solo 3 tienen la información relevante.
- Hay un **tope de seguridad** (`MAX_PAGINAS_HD`, default 12): si "todo" el documento puntuó positivo (contratos muy repetitivos donde la palabra "COP" aparece en cada página, por ejemplo), se toman solo las páginas con **mayor** puntaje en vez de procesar el documento entero en alta resolución.
- `extraer_texto_o_ocr` decide automáticamente si hace falta OCR: si el texto nativo tiene **menos de 500 caracteres o menos de 20 dígitos**, se asume que es un escaneo y se cae a OCR — un PDF de texto normal casi siempre supera ese umbral por un margen amplio, así que el umbral es conservador y no dispara OCR innecesariamente en documentos que sí tienen texto.

## Fase 3 — Proveedor DeepSeek (compatible con la API de OpenAI)

In [ ]:
class DeepSeekProvider:
    label = "deepseek"

    def __init__(self, model):
        import openai
        key = os.environ.get("DEEPSEEK_API_KEY")
        if not key:
            raise RuntimeError("Falta DEEPSEEK_API_KEY en el entorno.")
        self.client = openai.OpenAI(api_key=key, base_url="https://api.deepseek.com")
        self.model = model

    def complete_pdf(self, path_pdf: Path, prompt: str, max_tokens: int) -> str:
        texto, metodo = extraer_texto_o_ocr(path_pdf)
        if not texto.strip():
            raise RuntimeError(
                "No se pudo extraer texto ni por lectura nativa ni por OCR "
                "(archivo posiblemente corrupto o en blanco)."
            )

        nota_metodo = (
            "TEXTO EXTRAÍDO DIRECTAMENTE DEL PDF"
            if metodo == "nativo"
            else "TEXTO OBTENIDO POR OCR (puede tener errores de reconocimiento de caracteres, "
                 "revisa cifras y unidades con cautela extra)"
        )
        prompt_full = (
            prompt
            + f"\n\n{nota_metodo} (sin imágenes ni estructura de tabla original — "
              "interpreta con cautela cualquier tabla que se vea desalineada):\n"
            + texto[:LIMITE_CHARS_TEXTO_PDF]
        )

        resp = self.client.chat.completions.create(
            model=self.model,
            max_tokens=max_tokens,
            temperature=0,
            messages=[{"role": "user", "content": prompt_full}],
        )

        if hasattr(resp, "usage") and resp.usage:
            log.info(f"    tokens in={resp.usage.prompt_tokens} out={resp.usage.completion_tokens} | metodo={metodo}")

        return resp.choices[0].message.content


def build_provider():
    if PROVEEDOR != "deepseek":
        raise RuntimeError(f"Este script solo soporta PROVEEDOR='deepseek' (tienes '{PROVEEDOR}').")
    return DeepSeekProvider(model=MODELO)

A diferencia de la versión con Claude, aquí **no hay una capa de proveedores intercambiable** — este script está especializado en DeepSeek, precisamente porque la pieza de valor es la integración con el OCR local, no la abstracción entre proveedores. El aviso `nota_metodo` que se agrega al prompt es importante: cuando el texto viene de OCR, se le pide explícitamente al modelo que trate las cifras con más cautela, porque el reconocimiento de caracteres puede confundir dígitos similares (por ejemplo, "0" con "O", o "1" con "l").

## Fase 4 — Utilidades de archivo y validación

In [ ]:
def pdf_apto(path_pdf: Path) -> tuple:
    if not path_pdf.exists():
        return False, "archivo no existe"
    size_mb = path_pdf.stat().st_size / (1024 * 1024)
    if size_mb > MAX_TAMANO_MB:
        return False, f"demasiado grande ({size_mb:.1f} MB > {MAX_TAMANO_MB} MB)"
    if size_mb < 0.001:
        return False, "archivo vacío"
    return True, f"{size_mb:.2f} MB"


def codigo_desde_nombre(path_pdf: Path) -> str:
    """El nombre del archivo SIN extensión ni sufijo _EP/_AT es el código de contrato."""
    stem = path_pdf.stem.strip()
    return re.sub(r'_(EP|AT|OT)(_\d+)?$', '', stem)


def indicador_desde_ruta(path_pdf: Path, ruta_raiz: Path) -> str:
    """Si el PDF está en Descargas/{codigo_indicador}/archivo.pdf, extrae el código del indicador."""
    try:
        rel = path_pdf.relative_to(ruta_raiz)
        if len(rel.parts) > 1:
            return rel.parts[0]
    except ValueError:
        pass
    return ""

## Fase 5 — Llamada al modelo con reintentos

In [ ]:
def llamar_modelo_pdf(provider, path_pdf: Path, codigo_archivo: str) -> Optional[dict]:
    prompt = PROMPT_TEMPLATE.format(codigo_archivo=codigo_archivo)

    for intento in range(1, MAX_REINTENTOS + 1):
        try:
            raw = provider.complete_pdf(path_pdf, prompt, MAX_TOKENS_RESPUESTA)
            raw = raw.replace("```json", "").replace("```", "").strip()
            m = re.search(r"\{.*\}", raw, re.DOTALL)
            if m:
                raw = m.group(0)
            return json.loads(raw)

        except json.JSONDecodeError as e:
            log.warning(f"  JSON inválido (intento {intento}): {e}")
            time.sleep(2 * intento)

        except RuntimeError as e:
            # p.ej. "no se pudo extraer texto ni por lectura nativa ni por OCR" — no reintentar
            log.warning(f"  {e}")
            return None

        except Exception as e:
            msg = str(e).lower()
            if "insufficient" in msg or "balance" in msg or "credit" in msg:
                log.error("  SALDO INSUFICIENTE en DeepSeek — deteniendo ejecución.")
                raise SystemExit(1)
            elif "rate" in msg or "429" in msg:
                log.warning(f"  Rate limit [{provider.label}] (intento {intento}), esperando {PAUSA_RATE_LIMIT}s...")
                time.sleep(PAUSA_RATE_LIMIT)
            elif "400" in msg:
                log.warning(f"  Error 400 [{provider.label}]: {e} — no se reintenta (revisar archivo)")
                return None
            else:
                log.warning(f"  Error inesperado [{provider.label}] (intento {intento}): {type(e).__name__}: {e}")
                time.sleep(3 * intento)

    log.error(f"  Falla tras {MAX_REINTENTOS} intentos: {path_pdf.name}")
    return None

Misma lógica de reintentos diferenciados que la versión con Claude
(ver esa documentación para el detalle de cada rama), con una verificación
adicional específica de DeepSeek: si el mensaje de error contiene
`insufficient` / `balance` / `credit` (saldo agotado), el proceso **se
detiene por completo** en vez de solo marcar ese PDF como fallido — no tiene
sentido seguir intentando el resto del lote si la cuenta se quedó sin
saldo.

## Fase 6 — Excel: formato, guardado y reanudación

In [ ]:
def formatear_excel(path_excel: Path):
    wb = load_workbook(path_excel)
    ws = wb.active

    header_fill = PatternFill("solid", fgColor="1F4E78")
    header_font = Font(bold=True, color="FFFFFF", size=11)
    border = Border(left=Side(style="thin", color="CCCCCC"), right=Side(style="thin", color="CCCCCC"),
                     top=Side(style="thin", color="CCCCCC"), bottom=Side(style="thin", color="CCCCCC"))
    align_header = Alignment(horizontal="center", vertical="center", wrap_text=True)

    for cell in ws[1]:
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = align_header
        cell.border = border
    ws.row_dimensions[1].height = 36
    ws.freeze_panes = "A2"

    anchos = {
        "archivo_pdf": 32, "codigo_indicador": 14, "codigo_contrato_archivo": 26,
        "codigo_contrato": 22, "descripcion_contrato": 50, "municipio": 26,
        "departamento": 16, "subregion_impacto": 22, "year_contrato": 10,
        "precio_cop": 16, "moneda": 8, "cantidad": 11, "unidad_cantidad": 18,
        "costo_unitario_cop": 16, "componentes_costo": 40, "fuente_cantidad": 32,
        "confianza": 10, "observaciones": 38, "fecha_extraccion": 17, "proveedor_modelo": 20,
        "metodo_lectura": 14,
    }
    for i, col in enumerate(ws[1], start=1):
        ws.column_dimensions[get_column_letter(i)].width = anchos.get(col.value, 15)

    for i, col in enumerate(ws[1], start=1):
        if col.value in ("precio_cop", "costo_unitario_cop"):
            letra = get_column_letter(i)
            for cell in ws[letra][1:]:
                if cell.value is not None:
                    cell.number_format = '"$"#,##0'

    fills_conf = {"alta": PatternFill("solid", fgColor="E8F5E9"),
                  "media": PatternFill("solid", fgColor="FFF9C4"),
                  "baja": PatternFill("solid", fgColor="FFEBEE")}
    col_conf_idx = next((i for i, col in enumerate(ws[1], start=1) if col.value == "confianza"), None)
    if col_conf_idx:
        for row in ws.iter_rows(min_row=2):
            val = row[col_conf_idx - 1].value
            fill = fills_conf.get(str(val).lower() if val else None)
            if fill:
                for cell in row:
                    cell.fill = fill

    wb.save(path_excel)


ORDEN_COLUMNAS = [
    "archivo_pdf", "codigo_indicador", "codigo_contrato_archivo", "codigo_contrato",
    "descripcion_contrato", "municipio", "departamento", "subregion_impacto",
    "year_contrato", "precio_cop", "moneda", "cantidad", "unidad_cantidad",
    "costo_unitario_cop", "componentes_costo", "fuente_cantidad",
    "confianza", "observaciones", "metodo_lectura", "proveedor_modelo", "fecha_extraccion",
]


def guardar_excel(resultados: list, path: Path):
    df = pd.DataFrame(resultados)
    for c in ORDEN_COLUMNAS:
        if c not in df.columns:
            df[c] = None
    df = df[ORDEN_COLUMNAS]
    df.to_excel(path, index=False, sheet_name="Costeo_Unitario")
    try:
        formatear_excel(path)
    except Exception as e:
        log.warning(f"No se pudo aplicar formato al Excel: {e}")


def cargar_resultados_previos(path: Path) -> tuple:
    if not path.exists():
        return [], set()
    try:
        df_prev = pd.read_excel(path, sheet_name="Costeo_Unitario")
        if df_prev.empty or "archivo_pdf" not in df_prev.columns:
            return [], set()
        df_prev = df_prev.where(pd.notna(df_prev), None)
        resultados_previos = df_prev.to_dict(orient="records")
        procesados = set(df_prev["archivo_pdf"].dropna().astype(str).tolist())
        return resultados_previos, procesados
    except Exception as e:
        log.warning(f"No se pudo leer el Excel previo ({path.name}): {e}")
        log.warning("Se empezará desde cero.")
        return [], set()

Nótese la columna adicional **`metodo_lectura`** en `ORDEN_COLUMNAS`
(no está en la versión con Claude, donde no hace falta porque siempre lee el
PDF nativo). Esta columna existe en el esquema y en el formato del Excel
(`formatear_excel` le asigna ancho), pensada para registrar si cada fila se
extrajo con texto **nativo** u **OCR** — un dato útil para decidir qué filas
revisar con más cuidado. Vale la pena que el equipo la complete al armar la
fila de resultado en la Fase 7, ya que actualmente `DeepSeekProvider`
calcula el `metodo` pero esa información no se está trasladando al
diccionario final que se guarda en el Excel.

## Fase 7 — Proceso principal

En el script original esto se maneja con `argparse` (`--ruta-pdfs`,
`--salida`, `--limite`, `--rehacer`). En el notebook, `ejecutar_extraccion`
recibe los mismos parámetros como argumentos de función. También configura
la ruta de Tesseract antes de cualquier OCR.

In [ ]:
def ejecutar_extraccion(ruta_pdfs=RUTA_PDFS_DEFAULT, salida=EXCEL_SALIDA,
                         limite=None, rehacer=False):
    # configurar ruta de tesseract antes de cualquier OCR
    try:
        import pytesseract
        pytesseract.pytesseract.tesseract_cmd = RUTA_TESSERACT
        if not Path(RUTA_TESSERACT).exists():
            log.warning(f"⚠ No se encontró Tesseract en: {RUTA_TESSERACT}")
            log.warning("  Ajusta RUTA_TESSERACT en la configuración si algún PDF resulta escaneado.")
    except ImportError:
        log.warning("⚠ pytesseract no está instalado — el OCR fallará si se necesita.")
        log.warning("  Instala con: py -m pip install pymupdf pytesseract pillow --break-system-packages")

    print("=" * 70)
    print(f"  Extracción de costeo unitario — proveedor: {PROVEEDOR} ({MODELO})")
    print("  Lectura nativa del PDF cuando hay texto; OCR local (dos pasadas) si es escaneado.")
    print("=" * 70)

    try:
        provider = build_provider()
    except RuntimeError as e:
        log.error(str(e))
        log.error('  En PowerShell: $env:DEEPSEEK_API_KEY = "sk-..."')
        return

    ruta_raiz = Path(ruta_pdfs)
    if not ruta_raiz.exists():
        log.error(f"No existe la ruta de PDFs: {ruta_raiz}")
        return

    pdfs = sorted(set(ruta_raiz.rglob("*.pdf")))
    if not pdfs:
        log.error(f"No se encontraron PDFs en {ruta_raiz}")
        return

    excel_path = Path(salida)
    resultados, procesados = cargar_resultados_previos(excel_path)

    if procesados and not rehacer:
        pdfs_pendientes = [p for p in pdfs if p.name not in procesados]
        ya_procesados = len(pdfs) - len(pdfs_pendientes)
    else:
        pdfs_pendientes = pdfs
        ya_procesados = 0
        if rehacer:
            resultados, procesados = [], set()

    log.info("=" * 70)
    log.info(f"  Ruta de PDFs:    {ruta_raiz}")
    log.info(f"  PDFs detectados: {len(pdfs)}")
    if ya_procesados > 0:
        log.info(f"  Ya procesados:   {ya_procesados} (se omiten)")
        log.info(f"  Pendientes:      {len(pdfs_pendientes)}")
    log.info(f"  Proveedor:       {PROVEEDOR} | Modelo: {MODELO}")
    log.info(f"  Excel salida:    {salida}")
    log.info("=" * 70)

    if not pdfs_pendientes:
        log.info("Todos los PDFs ya están procesados. Nada por hacer (usa rehacer=True para reprocesar).")
        return

    pdfs = pdfs_pendientes
    if limite:
        pdfs = pdfs[:limite]
        log.info(f"Modo prueba: se procesarán solo {len(pdfs)} PDFs pendientes")

    total, exitos, fallos = len(pdfs), 0, 0

    for idx, pdf in enumerate(pdfs, start=1):
        codigo_archivo = codigo_desde_nombre(pdf)
        codigo_indicador = indicador_desde_ruta(pdf, ruta_raiz)
        log.info(f"\n[{idx}/{total}] {pdf.name}  (indicador={codigo_indicador or '?'}, contrato={codigo_archivo})")

        ok, info = pdf_apto(pdf)
        if not ok:
            log.warning(f"  Descartado: {info}")
            resultados.append({
                "archivo_pdf": pdf.name, "codigo_indicador": codigo_indicador,
                "codigo_contrato_archivo": codigo_archivo, "confianza": "baja",
                "observaciones": f"PDF descartado: {info}",
                "proveedor_modelo": f"{PROVEEDOR}:{MODELO}",
                "fecha_extraccion": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            })
            fallos += 1
            continue

        log.info(f"  Procesando ({info})...")
        extraccion = llamar_modelo_pdf(provider, pdf, codigo_archivo)

        if extraccion is None:
            log.warning("  Sin respuesta utilizable — NO se guarda (se reintentará en próxima ejecución)")
            fallos += 1
        else:
            fila = {
                "archivo_pdf": pdf.name, "codigo_indicador": codigo_indicador,
                "codigo_contrato_archivo": codigo_archivo,
                "codigo_contrato": extraccion.get("codigo_contrato"),
                "descripcion_contrato": extraccion.get("descripcion_contrato"),
                "municipio": extraccion.get("municipio"),
                "departamento": extraccion.get("departamento"),
                "subregion_impacto": extraccion.get("subregion_impacto"),
                "year_contrato": extraccion.get("year_contrato"),
                "precio_cop": extraccion.get("precio_cop"),
                "moneda": extraccion.get("moneda"),
                "cantidad": extraccion.get("cantidad"),
                "unidad_cantidad": extraccion.get("unidad_cantidad"),
                "costo_unitario_cop": extraccion.get("costo_unitario_cop"),
                "componentes_costo": extraccion.get("componentes_costo"),
                "fuente_cantidad": extraccion.get("fuente_cantidad"),
                "confianza": extraccion.get("confianza"),
                "observaciones": extraccion.get("observaciones"),
                "proveedor_modelo": f"{PROVEEDOR}:{MODELO}",
                "fecha_extraccion": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            }
            try:
                if (not fila["costo_unitario_cop"] and fila["precio_cop"] and fila["cantidad"]
                        and float(fila["cantidad"]) > 0):
                    fila["costo_unitario_cop"] = round(float(fila["precio_cop"]) / float(fila["cantidad"]))
            except (TypeError, ValueError):
                pass

            resultados.append(fila)
            exitos += 1
            log.info(
                f"  OK {fila.get('municipio') or '-'} | ${fila.get('precio_cop') or '-'} | "
                f"{fila.get('cantidad') or '-'} {fila.get('unidad_cantidad') or ''} | conf={fila.get('confianza')}"
            )

        time.sleep(PAUSA_ENTRE_LLAMADAS)

        if idx % CHECKPOINT_CADA == 0 or idx == total:
            guardar_excel(resultados, Path(salida))
            log.info(f"  [Checkpoint] {idx}/{total} guardados en {salida}")

    guardar_excel(resultados, Path(salida))

    log.info("\n" + "=" * 70)
    log.info("RESUMEN FINAL")
    log.info("=" * 70)
    log.info(f"  PDFs procesados:       {total}")
    log.info(f"  Extracciones exitosas: {exitos}")
    log.info(f"  Fallos:                {fallos}")

    df = pd.DataFrame(resultados)
    if not df.empty and "costo_unitario_cop" in df.columns:
        log.info(f"  Con costo unitario:    {df['costo_unitario_cop'].notna().sum()}")
        log.info(f"  Con cantidad:          {df['cantidad'].notna().sum()}")
        log.info(f"  Con precio:            {df['precio_cop'].notna().sum()}")
        for nivel in ["alta", "media", "baja"]:
            log.info(f"  Confianza '{nivel}': {(df['confianza'] == nivel).sum()}")

    log.info(f"\n  Archivo final: {Path(salida).resolve()}")
    log.info("=" * 70)

Nota importante: en `ejecutar_extraccion`, ante saldo insuficiente
de DeepSeek, `llamar_modelo_pdf` levanta `SystemExit(1)` — en un notebook
esto interrumpe la celda en vez de cerrar todo el kernel (a diferencia del
script original, donde `sys.exit(1)` termina el proceso completo). El
resultado práctico es el mismo: el proceso se detiene y no sigue gastando
saldo en el resto del lote.

## Fase 8 — Ejecución

Igual que en la versión con Claude: la celda queda lista para correr con la
configuración de arriba (`RUTA_PDFS_DEFAULT`, `LIMITE = None`). Ajustar
`RUTA_PDFS`, `LIMITE` y `REHACER` según la corrida que se quiera hacer, y
asegurarse de tener `DEEPSEEK_API_KEY` en el entorno y Tesseract instalado
si hay PDFs escaneados en el lote.

In [ ]:
RUTA_PDFS = RUTA_PDFS_DEFAULT   # -> cambiar a la carpeta real de descargas
SALIDA    = EXCEL_SALIDA
LIMITE    = None                # -> un número para procesar solo N PDFs de prueba
REHACER   = False                # -> True para ignorar el Excel existente y reprocesar todo

try:
    ejecutar_extraccion(ruta_pdfs=RUTA_PDFS, salida=SALIDA, limite=LIMITE, rehacer=REHACER)
except SystemExit:
    print("Ejecución detenida (ver log — probablemente saldo insuficiente en DeepSeek).")
except Exception as e:
    log.error(f"Fallo en la ejecución: {e}")
    if DETENER_EN_ERROR:
        raise